In [2]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
import math
# import mediapipe as mp
from datetime import datetime

In [4]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

NameError: name 'core' is not defined

In [7]:
# ==========================================
# 1. Configuration & Setup
# ==========================================
VIDEO_PATH = 'petrol-pump.mp4'  
OUTPUT_PATH = 'output_video_mediapipe.mp4'
LOG_FILE = 'activity_log_advanced.csv'

model = YOLO('yolov8n.pt')

# Initialize MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, min_detection_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

ROI_PUMP_LEFT = np.array([[380, 150], [480, 150], [480, 350], [380, 350]], np.int32)
ROI_PUMP_RIGHT = np.array([[550, 150], [680, 150], [680, 350], [550, 350]], np.int32)
ROIS = [ROI_PUMP_LEFT, ROI_PUMP_RIGHT]

track_history = {} 
MAX_HISTORY = 30  
STATIONARY_THRESHOLD = 20 
activity_logs = []

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [ ]:
# ==========================================
# 2. Helper Functions (Math & Geometry)
# ==========================================
def calculate_movement(history):
    """Calculates pixel distance moved over the stored history."""
    if len(history) < 2: return 0.0
    return math.dist(history[0], history[-1])

def check_in_roi(point, rois):
    """Checks if a coordinate is inside the workspace boundaries."""
    for roi in rois:
        if cv2.pointPolygonTest(roi, point, False) >= 0: return True
    return False

def calculate_angle(a, b, c):
    """
    Calculates the 2D angle between three points.
    Used for joint posture (e.g., Hip -> Knee -> Ankle).
    """
    a = np.array(a) # First
    b = np.array(b) # Mid (The joint we are measuring)
    c = np.array(c) # End
    
    radians = math.atan2(c[1]-b[1], c[0]-b[0]) - math.atan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

In [ ]:
# ==========================================
# 3. Main Processing Loop
# ==========================================
cap = cv2.VideoCapture(VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
        
    frame_count += 1
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Step 1: YOLO Tracking (Robust ID persistence)
    results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)

    for roi in ROIS:
        cv2.polylines(frame, [roi], isClosed=True, color=(255, 0, 0), thickness=2)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy() 
        track_ids = results[0].boxes.id.cpu().numpy() 
        classes = results[0].boxes.cls.cpu().numpy() 

        for box, track_id, cls in zip(boxes, track_ids, classes):
            if int(cls) != 0: continue # Process only persons
                
            x1, y1, x2, y2 = map(int, box)
            track_id = int(track_id)
            
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(width, x2), min(height, y2)
            
            if x2 - x1 < 20 or y2 - y1 < 20: continue

            # Step 2: Crop bounding box to pass to MediaPipe
            person_crop = frame[y1:y2, x1:x2]
            person_crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
            pose_results = pose.process(person_crop_rgb)

            # Default tracking values if MediaPipe fails to find a skeleton
            tracking_point = (int((x1 + x2) / 2), y2)
            is_bending = False

            if pose_results.pose_landmarks:
                landmarks = pose_results.pose_landmarks.landmark
                ch, cw, _ = person_crop.shape
                
                # --- A: Find the lowest foot for perfect ROI tracking ---
                left_ankle_y = landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].y * ch
                right_ankle_y = landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y * ch
                
                if left_ankle_y > right_ankle_y:
                    ankle_x = int(landmarks[mp_pose.PoseLandmark.LEFT_ANKLE.value].x * cw) + x1
                    ankle_y = int(left_ankle_y) + y1
                else:
                    ankle_x = int(landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x * cw) + x1
                    ankle_y = int(right_ankle_y) + y1
                    
                tracking_point = (ankle_x, ankle_y)

                # --- B: Calculate Posture Angles (Knee Bend) ---
                # Get coordinates for Right Hip, Knee, and Ankle
                r_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
                r_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
                r_ankle = [landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]
                
                # Calculate the angle at the knee
                knee_angle = calculate_angle(r_hip, r_knee, r_ankle)
                
                # If knee angle is less than ~160 degrees, they are bending/crouching slightly
                if knee_angle < 160:
                    is_bending = True

                # --- C: Draw the Skeleton ---
                mp_drawing.draw_landmarks(person_crop, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
                frame[y1:y2, x1:x2] = person_crop 

            # Step 3: Update tracking history
            if track_id not in track_history: track_history[track_id] = []
            track_history[track_id].append(tracking_point)
            
            if len(track_history[track_id]) > MAX_HISTORY:
                track_history[track_id].pop(0)

            # Step 4: Advanced Classification Logic Engine
            is_in_workspace = check_in_roi(tracking_point, ROIS)
            movement_distance = calculate_movement(track_history[track_id])
            is_stationary = movement_distance < STATIONARY_THRESHOLD

            # Incorporating the MediaPipe Posture into the logic
            if is_in_workspace and is_stationary:
                if is_bending:
                    state = "Working (Bending)"
                    color = (0, 255, 0) # Green
                else:
                    state = "Working (Standing)"
                    color = (255, 255, 0) # Yellow-ish
            elif not is_in_workspace and not is_stationary:
                state = "Moving"
                color = (0, 165, 255) # Orange
            else:
                state = "Idle"
                color = (0, 0, 255) # Red

            # Log to CSV
            if frame_count % fps == 0:
                activity_logs.append({
                    "Timestamp": current_time,
                    "Person_ID": track_id,
                    "Activity_State": state,
                    "In_ROI": is_in_workspace,
                    "Posture_Bending": is_bending
                })

            # Draw labels and trails
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID: {track_id} | {state}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            points = np.hstack(track_history[track_id]).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=(255, 255, 255), thickness=1)

    out.write(frame)

In [ ]:
# Clean up
cap.release()
out.release()
cv2.destroyAllWindows()
pose.close()

# Export logs
df = pd.DataFrame(activity_logs)
df.to_csv(LOG_FILE, index=False)
print("Advanced processing complete. CSV saved successfully!")